# AI Livestream Commerce VN — Colab Demo

T4 free primary, scales to L4/A100. Mock backend, no LiveAvatar cloud credits.


In [ ]:
# Setup: preflight, tier detect, and idempotent dependency install
import importlib.util
import os
import subprocess
import sys

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is required in Colab; enable a standard Python runtime.") from exc

def detect_tier():
    if not torch.cuda.is_available():
        return "cpu", "No GPU — demo will be slow; LLM may OOM. Use a GPU runtime."
    name = torch.cuda.get_device_name(0).upper()
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if "A100" in name:
        return "a100", f"A100 {mem_gb:.0f}GB — full presets, vLLM OK"
    if "L4" in name:
        return "l4", f"L4 {mem_gb:.0f}GB — vLLM OK, 7B fits"
    if "T4" in name:
        return "t4", f"T4 {mem_gb:.0f}GB — llamacpp 4B Q4 only"
    return "gpu", f"{name} {mem_gb:.0f}GB"

TIER, TIER_NOTE = detect_tier()
DESIRED_LLM_ENGINE = "vllm" if TIER in {"l4", "a100"} else "llamacpp"
print(f"Tier: {TIER} — {TIER_NOTE}")
print(f"Selected LLM engine: {DESIRED_LLM_ENGINE}")

def has_module(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def pip_install(*packages: str, quiet: bool = True) -> None:
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd.extend(packages)
    subprocess.check_call(cmd)

base_packages = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pyngrok",
    "httpx",
    "async_timeout",
    "pillow",
    "numpy",
    "huggingface_hub",
]
pip_install(*base_packages)

# llama-cpp-python: use the prebuilt CUDA wheel index so T4/L4/A100 get a
# GPU build without a slow from-source compile. Colab ships CUDA 12.x, so the
# cu121 wheels match. Fallback to a source build (no extra index) if the wheel
# index is unreachable.
_cuda_major = ""
try:
    _cuda_ver = torch.version.cuda or ""
    _cuda_major = _cuda_ver.split(".")[0] if _cuda_ver else "12"
except Exception:
    _cuda_major = "12"
_LLAMACPP_WHEEL_INDEX = (
    "https://abetlen.github.io/llama-cpp-python/whl/cu121"
    if _cuda_major == "12"
    else "https://abetlen.github.io/llama-cpp-python/whl/cu118"
)

def pip_install_llamacpp() -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "llama-cpp-python",
           "--extra-index-url", _LLAMACPP_WHEEL_INDEX]
    try:
        subprocess.check_call(cmd)
        print(f"Installed llama-cpp-python (CUDA {_cuda_major} wheel).")
    except subprocess.CalledProcessError:
        print("CUDA wheel install failed; falling back to source build (CPU).")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"])

if DESIRED_LLM_ENGINE == "vllm":
    pip_install("vllm")
    pip_install_llamacpp()
    print("Installed vLLM + llama-cpp-python for L4/A100 tier.")
else:
    pip_install_llamacpp()
    print("vLLM skipped for T4/free tier.")

INSTALL_VIENEU_OK = False
try:
    pip_install("neuttsair")
    INSTALL_VIENEU_OK = True
    print("Installed neuttsair for VieNeu TTS preset.")
except Exception as exc:
    print(f"VieNeu dependency install failed: {exc}")
    print("Falling back to transformers-mms-vi dependencies.")
    pip_install("transformers", "accelerate", "sentencepiece")

TTS_PRESET = "vieneu-v3-turbo" if INSTALL_VIENEU_OK else "transformers-mms-vi"
print(f"Selected TTS preset: {TTS_PRESET}")


In [ ]:
# Clone backend repository
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/ai-livestream-commerce-vn")
REPO_URL = "https://github.com/justHman/ai-livestream-commerce-vn.git"
REPO_BRANCH = "main"

if not (REPO_DIR / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--depth",
        "1",
        REPO_URL,
        str(REPO_DIR),
    ])
else:
    print(f"Repo already exists at {REPO_DIR}; fetching {REPO_BRANCH}.")
    subprocess.check_call(["git", "fetch", "origin", REPO_BRANCH, "--depth", "1"], cwd=str(REPO_DIR))
    subprocess.check_call(["git", "checkout", REPO_BRANCH], cwd=str(REPO_DIR))
    subprocess.check_call(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=str(REPO_DIR))

os.chdir(REPO_DIR)
print(f"Repo ready: {REPO_DIR} @ {REPO_BRANCH}")


In [ ]:
# Download GGUF for llama.cpp tiers; vLLM tiers pull from the Hub at runtime
import fnmatch
import os
from pathlib import Path

from huggingface_hub import hf_hub_download, list_repo_files

MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LLAMACPP_REPO_ID = "unsloth/Qwen3.5-4B-GGUF"
VLLM_MODEL_ID = "SeaLLMs/SeaLLMs-v3-7B-Chat"

if DESIRED_LLM_ENGINE == "llamacpp":
    existing = sorted(MODEL_DIR.glob("*Q4_K_M*.gguf"))
    if existing:
        MODEL_PATH = str(existing[0])
        print(f"Using existing GGUF: {MODEL_PATH}")
    else:
        files = list_repo_files(LLAMACPP_REPO_ID)
        matches = [
            name for name in files
            if fnmatch.fnmatch(Path(name).name, "*Q4_K_M*.gguf")
        ]
        if not matches:
            raise RuntimeError(f"No Q4_K_M GGUF found in {LLAMACPP_REPO_ID}")
        filename = sorted(matches)[0]
        MODEL_PATH = hf_hub_download(
            repo_id=LLAMACPP_REPO_ID,
            filename=filename,
            local_dir=str(MODEL_DIR),
        )
        print(f"Downloaded GGUF: {MODEL_PATH}")
else:
    MODEL_PATH = VLLM_MODEL_ID
    print(f"Skipping GGUF download for vLLM tier. vLLM model: {MODEL_PATH}")


In [ ]:
# Set backend env and launch uvicorn without blocking the notebook
import os
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = f"{REPO_DIR}:{os.environ.get('PYTHONPATH', '')}"
os.environ.update({
    "RENDER_BACKEND": "mock",
    "SESSION_STORE": "memory",
    "APP_ENV": "dev",
    "LLM_ENGINE": DESIRED_LLM_ENGINE,
    "LLM_MODEL_PATH": str(MODEL_PATH),
    "LLM_STREAM": "1",
    "LLM_N_GPU_LAYERS": "-1",
    "LLM_N_CTX": "4096",
    "TTS_PRESET_ID": TTS_PRESET,
    "DIRECTOR_ENABLED": "1",
    "DIRECTOR_EMBEDDER": "hash",
    "LIVEAVATAR_API_TOKEN": "",
    "ADMIN_API_TOKEN": "",
})

LOG_PATH = Path("/content/backend_uvicorn.log")
SERVER_PROC = globals().get("SERVER_PROC")
if SERVER_PROC is not None and SERVER_PROC.poll() is None:
    print(f"Server already running with PID {SERVER_PROC.pid}")
else:
    log_file = LOG_PATH.open("w", encoding="utf-8")
    SERVER_PROC = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "core.server:create_app",
            "--factory",
            "--host",
            "0.0.0.0",
            "--port",
            "8000",
        ],
        cwd=str(REPO_DIR),
        env=os.environ.copy(),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(f"Server starting with PID {SERVER_PROC.pid}")

print("Local URL: http://127.0.0.1:8000")
print(f"Log file: {LOG_PATH}")
print(f"LLM_ENGINE={os.environ['LLM_ENGINE']}")
print(f"LLM_MODEL_PATH={os.environ['LLM_MODEL_PATH']}")
print(f"TTS_PRESET_ID={os.environ['TTS_PRESET_ID']}")


In [ ]:
# Start ngrok tunnel
import os

from pyngrok import ngrok

if os.environ.get("NGROK_AUTHTOKEN"):
    ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

ngrok.kill()
tunnel = ngrok.connect(8000, bind_tls=True)
PUBLIC_URL = tunnel.public_url.rstrip("/")
print(f"Public URL: {PUBLIC_URL}")


In [ ]:
# Smoke test: health, session start, attach, chat, engines, and mock frame
import json
import time
from pathlib import Path

import httpx

BASE_URL = "http://127.0.0.1:8000"
API_PREFIXES = ["/api/v1", ""]

def endpoint(path: str, prefix: str = "/api/v1") -> str:
    return f"{BASE_URL}{prefix}{path}"

def request_first(method: str, path: str, **kwargs):
    last_error = None
    with httpx.Client(timeout=20.0) as client:
        for prefix in API_PREFIXES:
            try:
                response = client.request(method, endpoint(path, prefix), **kwargs)
                if response.status_code != 404:
                    return response, prefix
            except Exception as exc:
                last_error = exc
    if last_error is not None:
        raise last_error
    raise RuntimeError(f"No route found for {method} {path}")

ready_response = None
API_PREFIX = "/api/v1"
deadline = time.time() + 60
while time.time() < deadline:
    try:
        ready_response, API_PREFIX = request_first("GET", "/health/ready")
        if ready_response.status_code == 200:
            payload = ready_response.json()
            if payload.get("ok") is not False and payload.get("status") not in {"not_ready", "error"}:
                print("Ready:", json.dumps(payload, ensure_ascii=False, indent=2))
                break
    except Exception:
        pass
    time.sleep(2)
else:
    print("Backend did not become ready within 60 seconds. Last ready response:")
    if ready_response is not None:
        print(ready_response.status_code, ready_response.text[:1000])
    log_tail = Path("/content/backend_uvicorn.log").read_text(encoding="utf-8", errors="replace")[-4000:]
    print(log_tail)
    raise RuntimeError("Backend readiness check failed.")

start_response, API_PREFIX = request_first("POST", "/lite/start", json={"is_sandbox": True})
start_response.raise_for_status()
start_payload = start_response.json()
SID = start_payload["session_id"]
print("/lite/start:", json.dumps(start_payload, ensure_ascii=False, indent=2))

products = [
    {"id": "p1", "name": "Áo thun livestream cotton", "price": 99000},
    {"id": "p2", "name": "Son dưỡng hương đào", "price": 79000},
]
attach_response, _ = request_first("POST", "/lite/attach", json={"session_id": SID, "products": products})
attach_response.raise_for_status()
print("/lite/attach:", json.dumps(attach_response.json(), ensure_ascii=False, indent=2))

chat_response, _ = request_first("POST", "/lite/chat", json={
    "session_id": SID,
    "text": "giá bao nhiêu shop",
    "author": "khach_demo",
})
chat_response.raise_for_status()
print("/lite/chat:", json.dumps(chat_response.json(), ensure_ascii=False, indent=2))

engines_response, _ = request_first("GET", "/engines")
engines_response.raise_for_status()
engines_payload = engines_response.json()
print("/engines:", json.dumps(engines_payload, ensure_ascii=False, indent=2)[:3000])
if not engines_payload.get("tts", {}).get("loaded", True):
    print("WARNING: TTS did not report loaded; backend ToneEngine fallback should keep smoke running.")

frame_path = Path("/content/smoke_frame.png")
frame_response, _ = request_first("GET", f"/mock/frame/{SID}.png")
frame_response.raise_for_status()
frame_path.write_bytes(frame_response.content)
frame_size = frame_path.stat().st_size
print(f"Saved smoke frame: {frame_path} ({frame_size} bytes)")
assert frame_size > 5 * 1024, f"Frame is too small: {frame_size} bytes"

MJPEG_URL = f"{PUBLIC_URL}{API_PREFIX}/mock/video/{SID}.mjpeg"
FRAME_URL = f"{PUBLIC_URL}{API_PREFIX}/mock/frame/{SID}.png"
print("\nDemo URLs")
print(f"Backend URL: {PUBLIC_URL}")
print(f"Session ID: {SID}")
print(f"MJPEG URL: {MJPEG_URL}")
print(f"Frame URL: {FRAME_URL}")


In [ ]:
# P2 smoke metrics runner: HTTP-only end-to-end check against the running backend
import subprocess
import sys

smoke_cmd = [
    sys.executable,
    "-m",
    "core.debug.smoke",
    "--base-url",
    "http://127.0.0.1:8000",
    "--chat-count",
    "10",
    "--mjpeg-parts",
    "10",
    "--mjpeg-seconds",
    "3",
    "--json",
]
subprocess.check_call(smoke_cmd)

if globals().get("PUBLIC_URL"):
    print("\nPublic ngrok smoke (same checks through tunnel):")
    public_cmd = smoke_cmd.copy()
    public_cmd[public_cmd.index("http://127.0.0.1:8000")] = PUBLIC_URL
    subprocess.check_call(public_cmd)


## URLs + local frontend

Use the concrete URLs printed by the smoke-test cell. Open `liveavatar_api/frontend/lite.html` locally, paste the printed ngrok `Backend URL` into the Backend URL field, then click **Start → Attach → send chat**.

The mock demo surface is the MJPEG stream: `{PUBLIC_URL}/api/v1/mock/video/{sid}.mjpeg`. Mock mode does not use real LiveKit video; the LiveKit connection in `lite.html` may fail gracefully while the MJPEG `<img>` continues to show idle and utterance frames.


In [ ]:
# README / shutdown helpers
import httpx
from pyngrok import ngrok

README = f"""
AI Livestream Commerce VN — Colab Demo README

Backend URL: {globals().get('PUBLIC_URL', '<run ngrok cell first>')}
Session ID: {globals().get('SID', '<run smoke cell first>')}
MJPEG URL: {globals().get('MJPEG_URL', '<run smoke cell first>')}
Smoke frame: /content/smoke_frame.png
Backend log: /content/backend_uvicorn.log

How to use:
1. Open liveavatar_api/frontend/lite.html on your local machine.
2. Paste the Backend URL above.
3. Click Start, Attach, then send Vietnamese chat messages.
4. For the no-black-screen demo surface, open the MJPEG URL directly or let lite.html show it.

Known limitations:
- T4/free tier uses llama.cpp with a 4B Q4_K_M GGUF model only.
- L4/A100 tiers install vLLM and use a 7B HF model path.
- VieNeu install can fail in Colab; this notebook falls back to transformers-mms-vi dependencies.
- DIRECTOR_EMBEDDER=hash avoids sentence-transformers downloads but has lower clustering quality.
- Mock mode serves MJPEG, not real LiveKit video.

Shutdown:
- Call shutdown_demo() below to POST /lite/stop, terminate uvicorn, and kill ngrok tunnels.
"""
print(README)

def shutdown_demo() -> None:
    sid = globals().get("SID")
    prefix = globals().get("API_PREFIX", "/api/v1")
    if sid:
        try:
            response = httpx.post(
                f"http://127.0.0.1:8000{prefix}/lite/stop",
                json={"session_id": sid},
                timeout=10.0,
            )
            print(f"Stopped session {sid}: {response.status_code} {response.text[:500]}")
        except Exception as exc:
            print(f"Session stop warning: {exc}")
    proc = globals().get("SERVER_PROC")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except Exception:
            proc.kill()
        print("Uvicorn process stopped.")
    ngrok.kill()
    print("Ngrok tunnels closed.")

print("Run shutdown_demo() when you are done.")
